# AM5061 · Week 6 · Heat pump condenser

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "glide", "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
    "lmtd", "effectiveness", "ntu_required", "exergy", "T0_REF", "P0_REF",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


# ------------------------------------------------- exchanger relations
def lmtd(dT1, dT2):
    """Log-mean temperature difference.

    Falls back to the arithmetic mean when the two ends are within 1% of each
    other, where the log form is numerically unstable and the two agree to
    better than 0.01% anyway.
    """
    dT1, dT2 = float(dT1), float(dT2)
    if dT1 <= 0 or dT2 <= 0:
        raise ValueError(
            f"temperature difference must be positive at both ends "
            f"(got {dT1:g} and {dT2:g}). A non-positive end means the streams "
            "cross, which no exchanger of this configuration can do."
        )
    if abs(dT1 - dT2) < 0.01*max(dT1, dT2):
        return 0.5*(dT1 + dT2)
    return (dT1 - dT2)/math.log(dT1/dT2)


def effectiveness(config, NTU, Cr):
    """Effectiveness for the standard configurations.

    config: 'counter', 'parallel', 'shell1'  (one shell pass, 2/4/... tube passes),
            'cross-both-unmixed' (approximate), 'cross-Cmax-mixed', 'cross-Cmin-mixed'

    Cr = C_min/C_max. Cr = 0 is the phase-change limit and every configuration
    collapses to the same expression, which is why boilers and condensers are
    easy and everything else is not.
    """
    if NTU < 0:
        raise ValueError("NTU cannot be negative")
    if not 0 <= Cr <= 1:
        raise ValueError(f"Cr must be between 0 and 1, got {Cr:g}")
    if Cr == 0:                       # phase change on one side
        return 1 - math.exp(-NTU)
    if config == "counter":
        if abs(Cr - 1) < 1e-12:
            return NTU/(1 + NTU)
        e = math.exp(-NTU*(1 - Cr))
        return (1 - e)/(1 - Cr*e)
    if config == "parallel":
        return (1 - math.exp(-NTU*(1 + Cr)))/(1 + Cr)
    if config == "shell1":
        r = math.sqrt(1 + Cr*Cr)
        e = math.exp(-NTU*r)
        return 2/(1 + Cr + r*(1 + e)/(1 - e))
    if config == "cross-both-unmixed":
        return 1 - math.exp((math.exp(-Cr*NTU**0.78) - 1)*NTU**0.22/Cr)
    if config == "cross-Cmax-mixed":
        return (1/Cr)*(1 - math.exp(-Cr*(1 - math.exp(-NTU))))
    if config == "cross-Cmin-mixed":
        return 1 - math.exp(-(1 - math.exp(-Cr*NTU))/Cr)
    raise ValueError(f"unknown configuration {config!r}")


def ntu_required(config, eps, Cr, hi=200.0):
    """Invert effectiveness() for NTU. Design direction, rather than rating."""
    eps_max = effectiveness(config, hi, Cr)
    if eps >= eps_max:
        raise ValueError(
            f"effectiveness {eps:g} is unreachable for {config} at Cr={Cr:g}; "
            f"the limit as NTU->infinity is {eps_max:.6f}. "
            "Change the configuration or accept less."
        )
    return solve(lambda n: effectiveness(config, n, Cr) - eps, 1.0,
                 bracket=(1e-9, hi))


# --------------------------------------------------------------- exergy
T0_REF, P0_REF = 303.15, 101325.0      # 30 C, sea level: the Chennai dead state


def exergy(st, T0=T0_REF, p0=P0_REF):
    """Specific flow exergy, J/kg:  (h - h0) - T0*(s - s0).

    The dead state is the ambient the plant actually sits in, so it is a
    DESIGN CHOICE, not a constant. Report which one you used - a Chennai
    dead state and a European one give different answers for the same plant.
    """
    ref = State(st.fluid, T=T0, P=p0)
    return (st.h - ref.h) - T0*(st.s - ref.s)


---
## The case

The condenser of a **10 kW residential heat pump** on R134a. In Week 1 you were
handed the condensing temperature. Now you must produce the **tube length**.

Deliverable **D-6**: required length by **segment integration**, against the
same length by a **single mean-h LMTD** calculation. Quantify the error of the
lazy method.

### Why a condenser is three exchangers

Refrigerant enters superheated, condenses, and leaves subcooled. The
coefficient in those three zones differs by **an order of magnitude**. Averaging
across them is the mistake this case exists to expose.


## 1. The duty and the three zones

In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
from scipy.optimize import brentq
am.style_plots()

F = "R134a"
Q_H     = 10e3          # W    heating duty
T_cond  = am.K(45)      # K    condensing temperature
T_evap  = am.K(0)       # K
eta_s   = 0.70
dT_sub  = 5.0           # K    subcooling
T_w_in  = am.K(30)      # K    water on
dT_w    = 10.0          # K    water rise
D_i     = 0.008         # m    refrigerant tube bore
PI      = np.pi

p_cond, p_evap = am.p_sat(F, T_cond), am.p_sat(F, T_evap)
st1 = am.sat_vapour(F, p=p_evap)
h2s = PropsSI("H", "P", p_cond, "S", st1.s, F)
h2  = st1.h + (h2s - st1.h)/eta_s               # compressor discharge
st2 = am.State(F, P=p_cond, H=h2)
h_g = am.sat_vapour(F, p=p_cond).h              # start of condensation
h_f = am.sat_liquid(F, p=p_cond).h              # end of condensation
h3  = am.State(F, P=p_cond, T=T_cond - dT_sub).h

m_r = Q_H/(h2 - h3)                              # refrigerant flow
m_w = Q_H/(4180.0*dT_w)                          # water flow

zones = {"desuperheat": h2 - h_g, "condense": h_g - h_f, "subcool": h_f - h3}
print(f"  discharge {st2.T_C:.2f} C, superheat {st2.T_C - am.C(T_cond):.2f} K")
print(f"  refrigerant flow {m_r*1e3:.3f} g/s,  water flow {m_w:.4f} kg/s\n")
print(f"{'zone':14s}{'dh, kJ/kg':>12}{'Q, W':>10}{'% of duty':>11}")
for z, dh in zones.items():
    print(f"{z:14s}{dh/1e3:12.3f}{m_r*dh:10.1f}{100*dh/(h2-h3):11.1f}")


## 2. Coefficients, zone by zone

Single-phase zones get Dittus–Boelter. The condensing zone gets **Shah**, which
is a shear-driven (annular) correlation, and we check it against **Chato**,
which is the gravity-driven (stratified) alternative.

> **Provenance.** Shah's constants and Chato's 0.555 are quoted from standard
> usage. They were **not** verified against a primary source in the course
> library. Treat them as starting points.


In [ ]:
def h_dittus(m, D, fluid, p, h_bulk, cooling=True):
    """Single-phase turbulent, in-tube."""
    st = am.State(fluid, P=p, H=h_bulk)
    Re = 4*m/(PI*D*st.mu)
    Pr = st.cp*st.mu/st.k
    n  = 0.3 if cooling else 0.4
    return 0.023*max(Re,1)**0.8*Pr**n*st.k/D, Re

def h_shah(x, m, D, fluid, p):
    """Shah (1979). Annular / shear-driven in-tube condensation."""
    st_l = am.sat_liquid(fluid, p=p)
    Re_l = 4*m/(PI*D*st_l.mu)                 # all-liquid Reynolds
    Pr_l = st_l.cp*st_l.mu/st_l.k
    h_l  = 0.023*max(Re_l,1)**0.8*Pr_l**0.4*st_l.k/D
    p_r  = p/am.critical(fluid)["p"]
    return h_l*((1-x)**0.8 + 3.8*x**0.76*(1-x)**0.04/p_r**0.38)

def h_chato(dT, D, fluid, p):
    """Chato (1962). Stratified / gravity-driven, low vapour velocity."""
    st_l = am.sat_liquid(fluid, p=p); st_g = am.sat_vapour(fluid, p=p)
    hfg  = st_g.h - st_l.h
    hfg_p = hfg + 0.68*st_l.cp*max(dT, 1e-6)
    return 0.555*((st_l.d*(st_l.d-st_g.d)*9.80665*hfg_p*st_l.k**3)
                  /(st_l.mu*max(dT,1e-6)*D))**0.25

xs = np.linspace(0.02, 0.98, 60)
h_sh = [h_shah(x, m_r, D_i, F, p_cond) for x in xs]
h_ch = h_chato(5.0, D_i, F, p_cond)
print(f"  Shah ranges {min(h_sh):.0f} to {max(h_sh):.0f} W/m2K across quality")
print(f"  Chato (dT = 5 K, stratified)  {h_ch:.0f} W/m2K  - a single number,"
      "\n  because gravity-driven condensation does not care about quality.")


## 3. Segment integration

March along the condenser in enthalpy steps.

In [ ]:
h_water_side = 3000.0        # W/m2K, annulus side, taken as given here

def segment_march(N=300, use="shah"):
    """Walk from discharge to subcooled outlet, N equal-enthalpy steps."""
    h_pts = np.linspace(h2, h3, N+1)
    T_w   = T_w_in + dT_w                      # counterflow: water leaves at the hot end
    L_tot, rows = 0.0, []
    for i in range(N):
        hi, ho = h_pts[i], h_pts[i+1]
        h_mid  = 0.5*(hi + ho)
        dQ     = m_r*(hi - ho)
        st     = am.State(F, P=p_cond, H=h_mid)
        T_r    = st.T
        # which zone are we in?
        if h_mid > h_g:   zone, h_i = "desuperheat", h_dittus(m_r, D_i, F, p_cond, h_mid, True)[0]
        elif h_mid > h_f:
            zone = "condense"
            x = (h_mid - h_f)/(h_g - h_f)
            h_i = h_shah(x, m_r, D_i, F, p_cond) if use == "shah" \
                  else h_chato(max(T_r - (T_w - dT_w/2), 1.0), D_i, F, p_cond)
        else:             zone, h_i = "subcool", h_dittus(m_r, D_i, F, p_cond, h_mid, True)[0]
        U   = 1/(1/h_i + 1/h_water_side)        # thin copper wall neglected
        T_w_out_seg = T_w
        T_w -= dQ/(m_w*4180.0)                  # water cools as we march backwards
        dT1, dT2 = T_r - T_w_out_seg, T_r - T_w
        dTlm = am.lmtd(max(dT1,0.05), max(dT2,0.05))
        dL   = dQ/(U*PI*D_i*dTlm)
        L_tot += dL
        rows.append({"h, kJ/kg": h_mid/1e3, "zone": zone, "T_r, C": am.C(T_r),
                     "T_w, C": am.C(T_w), "h_i, W/m2K": h_i, "U, W/m2K": U,
                     "dT_lm, K": dTlm, "dL, m": dL, "L cumulative, m": L_tot})
    return L_tot, rows

L_seg, rows = segment_march()
by_zone = {}
for r_ in rows: by_zone[r_["zone"]] = by_zone.get(r_["zone"], 0.0) + r_["dL, m"]
print(f"  segment-integrated length  {L_seg:.3f} m\n")
print(f"{'zone':14s}{'length, m':>11}{'% of length':>13}{'% of duty':>11}")
for z in ("desuperheat","condense","subcool"):
    print(f"{z:14s}{by_zone[z]:11.3f}{100*by_zone[z]/L_seg:13.1f}"
          f"{100*zones[z]/(h2-h3):11.1f}")


Compare the two right-hand columns. Desuperheating takes about
10% of the duty but nearer 13% of the length, and subcooling likewise runs long
for its duty, because both are single-phase zones with poor coefficients. The
asymmetry is modest here; it grows sharply if the superheat is larger or the
water-side coefficient is worse.


## 4. The lazy method, and what it costs

In [ ]:
# One mean coefficient, one LMTD across the whole condenser.
h_mean = np.mean([r_["h_i, W/m2K"] for r_ in rows])
U_mean = 1/(1/h_mean + 1/h_water_side)
dT1 = st2.T - (T_w_in + dT_w)                       # hot end
dT2 = (T_cond - dT_sub) - T_w_in                    # cold end
L_lazy = Q_H/(U_mean*PI*D_i*am.lmtd(dT1, dT2))

print(f"  mean h              {h_mean:9.1f} W/m2K")
print(f"  single LMTD         {am.lmtd(dT1, dT2):9.3f} K")
print(f"  lazy length         {L_lazy:9.3f} m")
print(f"  segment-integrated  {L_seg:9.3f} m")
print(f"\n  error of the lazy method: {100*(L_lazy-L_seg)/L_seg:+.1f}%")
print("  Undersizing a condenser raises the condensing pressure, which raises")
print("  the discharge temperature and cuts the COP. It is not a safe error.")


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.8, 4.3))
L  = [r_["L cumulative, m"] for r_ in rows]
a1.plot(L, [r_["T_r, C"] for r_ in rows], lw=2.4, label="refrigerant")
a1.plot(L, [r_["T_w, C"] for r_ in rows], lw=2.4, label="water (counterflow)")
for z, col in (("desuperheat", "#FDEBD0"), ("condense", "#EAF2FB"), ("subcool", "#EAF7EF")):
    seg = [r_["L cumulative, m"] for r_ in rows if r_["zone"] == z]
    if seg: a1.axvspan(min(seg), max(seg), color=col, zorder=0)
a1.set_xlabel("length along condenser  (m)"); a1.set_ylabel("temperature  (°C)")
a1.set_title("Three zones, shaded"); a1.legend(fontsize=9)
a2.plot(L, [r_["h_i, W/m2K"] for r_ in rows], lw=2.4, color=am.ORANGE)
a2.axhline(h_mean, ls="--", color=am.MUTED)
a2.text(L[len(L)//2], h_mean*1.06, " the mean the lazy method uses", color=am.MUTED, fontsize=9)
a2.set_yscale("log"); a2.set_xlabel("length along condenser  (m)")
a2.set_ylabel("refrigerant-side h  (W/m²K)")
a2.set_title("An order of magnitude, averaged away")
plt.tight_layout(); plt.show()


## 5. Shah against Chato, and grid independence

In [ ]:
L_chato, _ = segment_march(use="chato")
print(f"  length with Shah  (shear-driven)   {L_seg:.3f} m")
print(f"  length with Chato (gravity-driven) {L_chato:.3f} m"
      f"   ({100*(L_chato-L_seg)/L_seg:+.1f}%)")
print("  Decide which regime you are in BEFORE opening a correlation.\n")
print(f"{'N':>6}{'length, m':>12}")
for N in (20, 50, 100, 300, 1000):
    print(f"{N:6d}{segment_march(N=N)[0]:12.4f}")
print("  Settles to about 20.2 m, but note it WOBBLES by a few tenths of a")
print("  percent rather than converging smoothly. That is because the zone")
print("  boundaries fall at different places inside a segment as N changes.")
print("  Refining a grid across a discontinuity does not converge cleanly, and")
print("  the honest fix is to put a node ON each boundary.")


## 6. The deliverable

In [ ]:
zone_rows = [{"zone": z, "Q, W": m_r*zones[z], "% of duty": 100*zones[z]/(h2-h3),
              "length, m": by_zone[z], "% of length": 100*by_zone[z]/L_seg}
             for z in ("desuperheat","condense","subcool")]
grid_rows = [{"N segments": N, "length, m": segment_march(N=N)[0]}
             for N in (20, 50, 100, 200, 300, 600, 1000)]

path = am.to_excel("AM5061_D6_Condenser.xlsx",
    {"Zone summary": zone_rows, "Marching profile": rows[::3], "Grid study": grid_rows},
    title="AM5061 D-6 . Condenser of a 10 kW residential heat pump",
    summary=[("Heating duty", Q_H, "W"), ("Refrigerant", F, ""),
             ("Condensing temperature", am.C(T_cond), "C"),
             ("Subcooling", dT_sub, "K"),
             ("Discharge temperature", st2.T_C, "C"),
             ("Refrigerant flow", m_r, "kg/s"), ("Water flow", m_w, "kg/s"),
             ("Length, segment integration", L_seg, "m"),
             ("Length, single mean-h LMTD", L_lazy, "m"),
             ("Error of the lazy method", 100*(L_lazy-L_seg)/L_seg, "%")],
    sources=[("R134a properties", "CoolProp 8.0.0"),
             ("Condensation, annular", "Shah (1979) - QUOTED, not source-verified here"),
             ("Condensation, stratified", "Chato (1962) - QUOTED, not source-verified here"),
             ("Single phase", "Dittus-Boelter, n = 0.3 for cooling"),
             ("Water-side coefficient", "3000 W/m2K, assumed")])
print("written:", path)


## What to hand in

1. The zone table: duty share and length share side by side.
2. The required tube length by segment integration.
3. The same by single mean-h LMTD, and **the error**, with a sentence on
   whether that error is safe or unsafe and why.
4. Shah against Chato, with a statement of which regime you believe you are in.
5. The workbook.

The water-side coefficient here is **assumed**, not computed. Week 9 removes
that assumption and turns this duty into actual hardware.
